# Part 3: PageRank on Spark— Iterative PageRank

In [1]:
from pyspark import SparkContext, SparkConf
import numpy as np

# Initialize Spark (use /DATA for temp storage since root filesystem is nearly full)
conf = SparkConf().setAppName("PageRank").setMaster("local[*]") \
    .set("spark.local.dir", "/DATA/suraj/m2/spark_tmp") \
    .set("spark.driver.memory", "2g")
sc = SparkContext.getOrCreate(conf=conf)

26/04/15 18:58:51 WARN Utils: Your hostname, dip resolves to a loopback address: 127.0.1.1; using 10.6.0.83 instead (on interface enp98s0f1)
26/04/15 18:58:51 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/04/15 18:58:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/15 18:58:52 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in mesos/standalone/kubernetes and LOCAL_DIRS in YARN).


In [2]:
DATA_DIR = "Q3- PageRank"
BETA = 0.8
ITERATIONS = 40

## PageRank Implementation


In [ ]:
def pagerank(sc, filepath, beta=0.8, iterations=40):
    #
    #Compute PageRank using Spark RDDs.
    
    #Args:
    #   sc: SparkContext,  filepath: path to edge list file (src dest per line)
    #    beta: damping factor, iterations: number of iterations
    
    #Returns:
    #    dict mapping node_id -> pagerank score
    
    import os
    
    # Read edges and deduplicate
    edges_rdd = sc.textFile(os.path.abspath(filepath)) \
        .map(lambda line: tuple(line.strip().split())) \
        .filter(lambda x: len(x) == 2) \
        .map(lambda x: (int(x[0]), int(x[1]))) \
        .distinct() \
        .cache()
    
    # Get all nodes (both source and destination)
    all_nodes = edges_rdd.flatMap(lambda x: [x[0], x[1]]).distinct().collect()
    n = len(all_nodes)
    print(f"Graph: {n} nodes, {edges_rdd.count()} unique edges")
    
    # Build adjacency list: source -> [list of destinations]
    adj_rdd = edges_rdd.groupByKey().mapValues(list).cache()
    
    # Initialize PageRank: r = 1/n for all nodes
    ranks = sc.parallelize([(node, 1.0 / n) for node in all_nodes])
    
    # Create zero-valued RDD for all nodes once (reused in rightOuterJoin)
    all_nodes_rdd = sc.parallelize([(node, 0.0) for node in all_nodes]).cache()
    
    teleport = (1.0 - beta) / n
    
    for iteration in range(iterations):
        # Join adjacency list with current ranks:
        # (node, ([neighbors], rank))
        contribs = adj_rdd.join(ranks).flatMap(
            lambda x: [(dest, x[1][1] / len(x[1][0])) for dest in x[1][0]]
        )
        
        # Sum contributions for each node and apply teleport
        ranks = contribs.reduceByKey(lambda a, b: a + b) \
            .rightOuterJoin(all_nodes_rdd) \
            .mapValues(lambda x: teleport + beta * (x[0] if x[0] is not None else 0.0))
    
    # Collect results
    result = dict(ranks.collect())
    
    edges_rdd.unpersist()
    adj_rdd.unpersist()
    all_nodes_rdd.unpersist()
    
    return result

## Experiment on Small Graph


In [4]:
import os

# Run on small graph
small_path = os.path.join(DATA_DIR, "small.txt")
ranks_small = pagerank(sc, small_path, beta=BETA, iterations=ITERATIONS)

# Sort by rank descending
sorted_small = sorted(ranks_small.items(), key=lambda x: x[1], reverse=True)

print(f"\nTop 5 nodes (small graph):")
for node, score in sorted_small[:5]:
    print(f"  Node {node}: {score:.6f}")

print(f"\nBottom 5 nodes (small graph):")
for node, score in sorted_small[-5:]:
    print(f"  Node {node}: {score:.6f}")

print(f"\nTop score: {sorted_small[0][1]:.3f} (expected ≈ 0.036)")

Graph: 100 nodes, 950 unique edges



Top 5 nodes (small graph):
  Node 53: 0.035731
  Node 14: 0.034171
  Node 40: 0.033630
  Node 1: 0.030006
  Node 27: 0.029720

Bottom 5 nodes (small graph):
  Node 89: 0.003922
  Node 37: 0.003808
  Node 81: 0.003695
  Node 59: 0.003670
  Node 85: 0.003410

Top score: 0.036 (expected ≈ 0.036)


In [5]:
# Run on whole graph
whole_path = os.path.join(DATA_DIR, "whole.txt")
ranks_whole = pagerank(sc, whole_path, beta=BETA, iterations=ITERATIONS)

# Sort by rank descending
sorted_whole = sorted(ranks_whole.items(), key=lambda x: x[1], reverse=True)

print(f"\nTop 5 nodes with highest PageRank:")
for node, score in sorted_whole[:5]:
    print(f"  Node {node}: {score:.6f}")

print(f"\nBottom 5 nodes with lowest PageRank:")
for node, score in sorted_whole[-5:]:
    print(f"  Node {node}: {score:.6f}")

print(f"\nSum of all PageRank scores: {sum(ranks_whole.values()):.6f} (should be ≈ 1.0)")

Graph: 1000 nodes, 8161 unique edges



Top 5 nodes with highest PageRank:
  Node 263: 0.002020
  Node 537: 0.001943
  Node 965: 0.001925
  Node 243: 0.001853
  Node 285: 0.001827

Bottom 5 nodes with lowest PageRank:
  Node 408: 0.000388
  Node 424: 0.000355
  Node 62: 0.000353
  Node 93: 0.000351
  Node 558: 0.000329

Sum of all PageRank scores: 1.000000 (should be ≈ 1.0)


In [6]:
# Clean up
sc.stop()